# 🚀 rues-linker — Notebook de ejecución

**Patrón:** dos celdas — `EXTRAS` (setup una vez) + `EJECUTAR` (≤15 líneas).

**Pre-requisitos:**
1. Paquete instalado: `!pip install -q git+https://github.com/enriqueforero/rues-linker.git`
2. Fuentes en `data/input/` con nombres `fuente_RUES.parquet`, `fuente_DIAN.parquet`, etc.
3. Credenciales Snowflake en Colab Secrets (icono 🔑 a la izquierda). Ver `docs/secrets.md`.

## 📦 CELDA 1 — EXTRAS / SETUP (ejecutar UNA vez al iniciar la sesión)

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELDA EXTRAS — imports y helpers. UNA SOLA VEZ por sesión.
# NO hay monkey-patches: el Orchestrator selecciona engine automáticamente.
# ════════════════════════════════════════════════════════════════════

from pathlib import Path
import pandas as pd
import psutil

from record_linkage.config import Config, Rutas
from record_linkage.config.profiles import config_produccion_it7
from record_linkage.pipeline.orchestrator import Orchestrator
from record_linkage.utils.colombia_time import hora_colombia


def cargar_fuentes(input_dir: Path) -> dict:
    """Carga fuente_<NOMBRE>.parquet desde input_dir."""
    fuentes = {}
    for nombre in ['RUES', 'DIAN', 'CRM', 'SUPERSOCIEDADES']:
        ruta = input_dir / f'fuente_{nombre}.parquet'
        if ruta.exists():
            fuentes[nombre] = pd.read_parquet(ruta)
            print(f'   ✅ {nombre}: {len(fuentes[nombre]):,} registros')
        else:
            print(f'   ⚠️  {nombre}: NO ENCONTRADA en {ruta}')
    return fuentes


print(f'✅ Setup completo — {hora_colombia()}')
print(f'   • Memoria libre: {psutil.virtual_memory().available / (1024**3):.1f} GB')

## 🚀 CELDA 2 — EJECUTAR (≤ 15 líneas, modifique solo el bloque CONFIG)

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────
WORKSPACE = '/content/drive/MyDrive/rues_linker_data'
ITERACION = 'IT8_PROD'
TRUSTED = {'RUES', 'SUPERSOCIEDADES'}

# ── EJECUCIÓN ────────────────────────────────────────────────────────
cfg = Config(workspace=WORKSPACE, iteracion=ITERACION, trusted_sources=TRUSTED)
rutas = Rutas.desde_config(cfg); rutas.crear_directorios()

fuentes = cargar_fuentes(rutas.entrada)
config_orch = {**config_produccion_it7}
config_orch['profiles']['enterprise_scale_4_sources']['trusted_unique_sources'] = sorted(TRUSTED)

orch = Orchestrator(config=config_orch, sources=fuentes, work_dir=str(rutas.base))
resultado = orch.run()

print(f'✅ Golden: {len(resultado["golden"]):,} · Correlativa: {len(resultado["correlative"]):,}')

## 📊 (Opcional) Inspección de resultados

In [ ]:
golden = resultado['golden']
correlativa = resultado['correlative']

print('=== GOLDEN RECORDS ===')
print(f'Shape: {golden.shape}')
print(f'Columnas: {list(golden.columns)}')
golden.head(10)

In [ ]:
# Exportar a Parquet
from datetime import datetime
ts = datetime.now().strftime('%Y%m%d_%H%M%S')

golden.to_parquet(rutas.salida / f'golden_{ITERACION}_{ts}.parquet', index=False)
correlativa.to_parquet(rutas.salida / f'correlativa_{ITERACION}_{ts}.parquet', index=False)

print(f'💾 Exportado a: {rutas.salida}')